# Notebook da Tarefa 2 — Baseline do Fluxo e Rotulagem (OK, RISCO, FALHA)

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Disciplina:** Estrutura de Dados II / Redes de Computadores  
**Equipe:** Grupo 42  
**Integrantes:**  
- Wendell Rodrigues Macedo  
- Vinicius Firmino Souto  
- Fernando Paiva Coelho (Scrum Master da Tarefa)  
- João Gustavo Sanches  
- Daniel Dias Paranhos  

---

### Objetivo e Contrato da Tarefa 2
- **Entrada:** `data/raw/` da Tarefa 1 (preservado e intocado).
- **Processamento:** Extração da ficha de baseline congelada no Período A (com piso de 1.500 amostras válidas), cálculo de métricas relativas no Período B e rotulagem sequencial (FALHA > RISCO > OK).
- **Saídas obrigatórias:**
  1. `baseline_por_fluxo.csv` (ficha de baseline congelada por fluxo).
  2. `dataset_rotulado_periodo_b.csv` (dataset com métricas relativas e rótulo auditável).
  3. Recorte temporal causal em B (50% Treino, 20% Validação, 30% Teste — sem shuffle aleatório).
  4. Dicionário de dados v0.2 em `metadados/dicionario_dados_v0.2.md`.
  5. Diário preenchido em `tarefas/Tarefa2_Baseline_e_Rotulagem.md`.
- **Regra de Ouro:** A árvore **não** entra aqui. País, IP, `rota_id`, `fluxo_id` e RTT absoluto **não** farão parte dos atributos da futura árvore.

## 1. Configuração do Ambiente e Bibliotecas

In [1]:
import os
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Definição e estruturação de diretórios do projeto
DIR_PROJETO = Path(".").resolve()
DIR_RAW = DIR_PROJETO / "data" / "raw"
DIR_INTERIM = DIR_PROJETO / "data" / "interim"
DIR_PROCESSED = DIR_PROJETO / "data" / "processed"
DIR_METADADOS = DIR_PROJETO / "metadados"
DIR_TAREFAS = DIR_PROJETO / "tarefas"

for d in [DIR_INTERIM, DIR_PROCESSED, DIR_METADADOS, DIR_TAREFAS]:
    d.mkdir(parents=True, exist_ok=True)

print("Diretórios configurados com sucesso:")
print(f"Raw:       {DIR_RAW}")
print(f"Interim:   {DIR_INTERIM}")
print(f"Processed: {DIR_PROCESSED}")
print(f"Metadados: {DIR_METADADOS}")

Diretórios configurados com sucesso:
Raw:       /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/raw
Interim:   /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/interim
Processed: /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/processed
Metadados: /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/metadados


## 2. Inspeção do Dado Bruto (`data/raw/`)
O dado bruto permanece rigorosamente intocado. A auditoria verifica contagens, integridade de timestamps, nulos e conformidade de timeouts (proibido RTT ausente salvo como 0 ms).

In [2]:
# Localizar arquivos na pasta data/raw
arquivos_raw = list(DIR_RAW.glob("*.csv"))
print(f"Arquivos CSV encontrados em raw: {[f.name for f in arquivos_raw]}")

arquivos_14d = list(DIR_RAW.glob("*14dias*.csv"))
if arquivos_14d:
    caminho_csv = arquivos_14d[0]
elif arquivos_raw:
    caminho_csv = arquivos_raw[0]
else:
    caminho_csv = None

if caminho_csv:
    print(f"Carregando {caminho_csv.name}...")
    df_raw = pd.read_csv(caminho_csv, low_memory=False)
    print(f"Dimensões do dataset bruto: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
else:
    print("Nenhum arquivo CSV encontrado em data/raw!")

Arquivos CSV encontrados em raw: ['ripe_atlas_14dias_20260920T043200Z.csv', 'ripe_atlas_m1001_20260916T015838Z.csv']
Carregando ripe_atlas_14dias_20260920T043200Z.csv...
Dimensões do dataset bruto: 336326 linhas x 16 colunas


In [3]:
# Auditoria de Integridade dos Dados Brutos
print("--- AUDITORIA DE INTEGRIDADE ---")
print(f"Total de linhas brutas: {len(df_raw)}")
print(f"Sondas únicas (prb_id): {df_raw['prb_id'].nunique()}")

# Chave de fluxo estável: probe_id | dst_addr
df_raw['fluxo_id'] = df_raw['prb_id'].astype(str) + "|" + df_raw['dst_addr'].astype(str)
print(f"Fluxos únicos (fluxo_id): {df_raw['fluxo_id'].nunique()}")

# Verificação de timeouts: latência média não pode ser 0 ms para ausências
tem_zero_invalido = (df_raw['avg'] == 0).any()
print(f"Existe latência igual a 0 ms gravada erroneamente? {tem_zero_invalido}")
print(f"Medições com avg nulo (timeouts/sem resposta): {df_raw['avg'].isna().sum()}")

--- AUDITORIA DE INTEGRIDADE ---
Total de linhas brutas: 336326
Sondas únicas (prb_id): 18
Fluxos únicos (fluxo_id): 67
Existe latência igual a 0 ms gravada erroneamente? False
Medições com avg nulo (timeouts/sem resposta): 19964
Duplicatas (fluxo_id, timestamp): 0


## 3. Delimitação dos Períodos Temporais (Período A e Período B)
- **Período A (Baseline):** bloco temporal anterior, utilizado unicamente para estimar o comportamento normal congelado de cada fluxo.
- **Período B (Dataset de Rotulagem):** bloco temporal subsequente, sem sobreposição de amostras com o Período A.

In [4]:
# Ordenação temporal estrita por timestamp
df_raw['timestamp'] = pd.to_numeric(df_raw['timestamp'])
df_raw = df_raw.sort_values(by=['fluxo_id', 'timestamp']).reset_index(drop=True)

print("Intervalo temporal do bruto:")
print(f"Timestamp mínimo (UTC): {pd.to_datetime(df_raw['timestamp'].min(), unit='s', utc=True)}")
print(f"Timestamp máximo (UTC): {pd.to_datetime(df_raw['timestamp'].max(), unit='s', utc=True)}")

Intervalo temporal do bruto:
Timestamp mínimo (UTC): 2026-09-06 04:32:00+00:00
Timestamp máximo (UTC): 2026-09-20 04:28:00+00:00
Período A (Baseline):  2026-09-06 04:32:00 UTC até 2026-09-13 04:32:00 UTC (167.962 medições)
Período B (Dataset):   2026-09-13 04:32:00 UTC até 2026-09-20 04:32:00 UTC (168.364 medições)
Timestamps compartilhados no mesmo fluxo entre A e B: 0


## 4. Cálculo da Ficha de Baseline por Fluxo (Somente Período A)

Para cada `fluxo_id`, calculamos com base estrita no Período A:
- `mediana`: mediana dos RTT válidos (ms)
- `MAD`: mediana(|RTT - mediana|)
- `jitter_tipico`: mediana do jitter nas medições válidas (ms)
- `perda_tipica`: mediana da porcentagem de perda (%)
- `prop_resposta`: proporção de medições com RTT válido sobre o total do período

> **Regra do Piso:** fluxos com menos de 1.500 RTT válidos no Período A são classificados como `baseline_insuficiente` e excluídos da rotulagem.

In [ ]:
# Parâmetro do piso de amostras válidas no Período A
PISO_AMOSTRAS_VALIDAS = 1500
print(f"Piso amostral exigido no Período A: {PISO_AMOSTRAS_VALIDAS} RTTs válidos")

## 5. Cálculo das Métricas Relativas no Período B

Com a ficha de baseline congelada de cada fluxo, calculamos para cada medição de B:
- `z_robusto = (RTT - mediana) / (1.4826 * MAD)`
- `aumento_pct = (RTT - mediana) / mediana * 100`
- `jitter_relativo = jitter / jitter_tipico`
- `perda_pct = (sent - rcvd) / sent * 100`
- `timeout_atual = 1` se RTT ausente ou perda=100%, senão `0`
- `n5_timeout`: timeouts nas últimas 5 medições deste fluxo (0 a 5)
- `n5_aumento80`: quantas das últimas 5 têm aumento_pct > 80% (0 a 5)
- `n5_risco`: quantas das últimas 5 cumprem a regra de risco (0 a 5)

In [ ]:
COLUNAS_RELATIVAS = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]
print(f"Métricas relativas a serem calculadas: {COLUNAS_RELATIVAS}")

## 6. Aplicação da Tabela de Rotulagem Hierárquica

A classe nasce de uma política determinística e auditável. Para cada medição do Período B de fluxos válidos, parar na **primeira** condição verdadeira:

| Ordem | Classe | Métrica | Limiar exato |
|---|---|---|---|
| 1 | **FALHA** | `perda_pct` | ≥ 10% nesta medição (1 perda em 3 pacotes = 33%) |
| 2 | **FALHA** | `n5_timeout` | ≥ 3 |
| 3 | **FALHA** | `z_robusto` | ≥ 3,5 nesta medição, com RTT presente |
| 4 | **FALHA** | `n5_aumento80` | ≥ 2 |
| 5 | **RISCO** | `(2 <= z_robusto < 3.5 OU 30 <= aumento_pct <= 80 OU jitter_relativo >= 3)` | e `n5_risco` ≥ 2 |
| 6 | **OK** | nenhuma das anteriores | pico isolado é OK |

In [ ]:
def classificar_medicao(linha):
    """Aplica a tabela de decisão na ordem estrita 1 a 6."""
    # Ordem 1: Perda >= 10%
    if linha.get("perda_pct", 0) >= 10.0:
        return "FALHA"
    # Ordem 2: Timeouts persistentes recentes
    if linha.get("n5_timeout", 0) >= 3:
        return "FALHA"
    # Ordem 3: Desvio robusto extremo
    z = linha.get("z_robusto")
    if z is not None and not np.isnan(z) and z >= 3.5:
        return "FALHA"
    # Ordem 4: Aumento substancial persistente
    if linha.get("n5_aumento80", 0) >= 2:
        return "FALHA"
    # Ordem 5: Risco persistente (n5_risco >= 2)
    cond_risco_atual = False
    if z is not None and not np.isnan(z) and 2.0 <= z < 3.5:
        cond_risco_atual = True
    aum = linha.get("aumento_pct")
    if aum is not None and not np.isnan(aum) and 30.0 <= aum <= 80.0:
        cond_risco_atual = True
    jit_rel = linha.get("jitter_relativo")
    if jit_rel is not None and not np.isnan(jit_rel) and jit_rel >= 3.0:
        cond_risco_atual = True
    if cond_risco_atual and linha.get("n5_risco", 0) >= 2:
        return "RISCO"
    # Ordem 6: Normal (OK)
    return "OK"

print("Função de rotulagem determinística compilada com sucesso.")

## 7. Recorte Temporal Causal de B (Treino 50%, Validação 20%, Teste 30%)

A separação é realizada respeitando a seta do tempo (passado treina o futuro):
- **Treino (50%):** bloco mais antigo do Período B.
- **Validação (20%):** bloco intermediário.
- **Teste (30%):** bloco mais recente.
> **Atenção:** Proibido uso de amostragem aleatória (`shuffle=True`).

In [ ]:
print("Estrutura de split temporal: 50% Treino / 20% Validação / 30% Teste por fluxo")

## 8. Exportação dos Artefatos Finais e Dicionário v0.2

In [ ]:
print("Pipeline da Tarefa 2 pronto para execução passo a passo.")